# Imports

In [1]:
import torch
import torch.nn as nn
import os
from torchvision import transforms
from PIL import Image
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm
import torchvision.transforms.functional as TF
from torch.optim.lr_scheduler import StepLR
import torch.nn.init as init
import matplotlib.pyplot as plt

# Dataloading

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anshtanwar/pets-facial-expression-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/pets-facial-expression-dataset


In [3]:
os.listdir(path)

['Other', 'Angry', 'Sad', 'Master Folder', 'happy']

In [4]:
master_path= os.path.join(path, 'Master Folder')

In [5]:
train_path= os.path.join(master_path, 'train')
val_path= os.path.join(master_path, 'valid')
test_path= os.path.join(master_path, 'test')

In [6]:
def get_data_paths(path):
  paths= []
  labels = os.listdir(path)
  for label in labels:
    label_path= os.path.join(path, label)
    for img in os.listdir(label_path):
      img_path= os.path.join(label_path, img)
      paths.append((img_path, label))
  return paths


In [7]:
train_paths= get_data_paths(train_path)
val_paths= get_data_paths(val_path)
test_paths= get_data_paths(test_path)

In [8]:
train_paths= np.array(train_paths)
val_paths= np.array(val_paths)
test_paths= np.array(test_paths)

In [9]:
def map_label(label):
  if label == 'happy':
    return 0
  elif label == 'Sad':
    return 1
  elif label == 'Angry':
    return 2
  else:
    return 3

In [10]:
def demap_label(label):
    if label == 0:
        return 'happy'
    elif label == 1:
        return 'Sad'
    elif label == 2:
        return 'Angry'
    else:
        return 'Other'

In [11]:
class ImgDataset(Dataset):
    def __init__(self, paths, transform= None):
      self.paths= paths[:, 0]
      self.labels= paths[:, 1]
      self.transform= transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, map_label(label)

In [12]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Resize the image to 224x224
    transforms.ToTensor(),              # Convert PIL Image to PyTorch tensor
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize
])

In [13]:
train_dataset= ImgDataset(train_paths, transform= transform)
val_dataset= ImgDataset(val_paths, transform= transform)
test_dataset= ImgDataset(test_paths, transform= transform)

In [14]:
batch_size= 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [15]:
# Function to display a batch of images and labels
def show_images(images, labels, num_images=16):
    """
    Displays images in a grid with their labels.
    Args:
    - images: Batch of images (Tensor).
    - labels: Corresponding labels (Tensor or list).
    - num_images: Number of images to display.
    """
    images = images[:num_images]  # Select the number of images to display
    labels = labels[:num_images]  # Select corresponding labels

    # Determine grid size
    grid_size = int(num_images ** 0.5)
    fig, axs = plt.subplots(grid_size, grid_size, figsize=(8, 8))

    for i, ax in enumerate(axs.flatten()):
        if i < len(images):
            # Convert tensor to a numpy array for visualization
            image = images[i].permute(1, 2, 0).cpu().numpy()  # CHW -> HWC
            ax.imshow(image)
            ax.set_title(f"Label: {demap_label(labels[i])}")
            ax.axis('off')
        else:
            ax.axis('off')

    plt.tight_layout()
    plt.show()

In [16]:
for images, labels in train_loader:
    #show_images(images, labels)
    pass

In [17]:
def augment_imgs(imgs, labels):
    augmented_imgs = []
    augmented_labels = []

    for img, label in zip(imgs, labels):
        # Original image
        augmented_imgs.append(img)
        augmented_labels.append(label)

        # Horizontal flip
        img_hflip = TF.hflip(img)
        augmented_imgs.append(img_hflip)
        augmented_labels.append(label)

        # Vertical flip
        img_vflip = TF.vflip(img)
        augmented_imgs.append(img_vflip)
        augmented_labels.append(label)

        # Combined flip (horizontal + vertical)
        img_hvflip = TF.hflip(img_vflip)
        augmented_imgs.append(img_hvflip)
        augmented_labels.append(label)

    # Convert lists to tensors
    augmented_imgs = torch.stack(augmented_imgs)
    augmented_labels = torch.tensor(augmented_labels)

    return augmented_imgs, augmented_labels

# Model

In [18]:
class VGG16(nn.Module):
    def __init__(self, num_classes=4):
        super(VGG16, self).__init__()
        # Define the convolutional layers
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Define the fully connected layers
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
            )
        self._initialize_weights()

    def _initialize_weights(self):
        # Apply Xavier initialization to convolutional layers and fully connected layers
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                init.xavier_normal_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                init.ones_(m.weight)
                init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        x = self.classifier(x)
        return x


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VGG16(num_classes=4).to(device)

In [20]:
for name, module in model.named_modules():
    print(name, module)

 VGG16(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, threshold= 1e-3)

In [ ]:
num_epochs= 30
train_losses=[]
train_accuracies=[]
val_losses=[]
val_accuracies=[]
# Training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Training step
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels= augment_imgs(images, labels)
        images, labels = images.to(device), labels.to(device)  # Move data to device

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)
    
    train_accuracy = 100 * correct_train / total_train
    train_losses.append(train_loss/len(train_loader))
    train_accuracies.append(train_accuracy) 
    print(f"Training Loss: {train_loss/len(train_loader):.4f}, Training Accuracy: {train_accuracy:.2f}%")

    # Validation step
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    scheduler.step(val_loss/len(val_loader))
    val_accuracy = 100 * correct_val / total_val
    val_losses.append(val_loss/len(val_loader))
    val_accuracies.append(val_accuracy) 
    print(f"Validation Loss: {val_loss/len(val_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}")
    print()


# Save the trained model
torch.save(model.state_dict(), "vgg16_trained.pth")


Epoch 1/30: 100%|██████████| 63/63 [00:37<00:00,  1.67it/s]


Training Loss: 23.5085, Training Accuracy: 59.98%
Validation Loss: 6.2909, Validation Accuracy: 30.56%
Current Learning Rate: 0.0001



Epoch 2/30: 100%|██████████| 63/63 [00:37<00:00,  1.68it/s]


Training Loss: 2.4776, Training Accuracy: 43.67%
Validation Loss: 6.2263, Validation Accuracy: 30.56%
Current Learning Rate: 0.0001



Epoch 3/30: 100%|██████████| 63/63 [00:37<00:00,  1.69it/s]


Training Loss: 1.8407, Training Accuracy: 37.52%
Validation Loss: 7.3015, Validation Accuracy: 30.56%
Current Learning Rate: 0.0001



Epoch 4/30:  46%|████▌     | 29/63 [00:17<00:20,  1.67it/s]

In [ ]:
print(train_losses)
print(train_accuracies)
print(val_losses)
print(val_accuracies)

In [ ]:
 # Test step
model.eval()  # Set model to evaluation mode
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Track loss and accuracy
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_test += (predicted == labels).sum().item()
        total_test += labels.size(0)

        print(labels)
        print(predicted)

test_accuracy = 100 * correct_test / total_test
print(f"test Loss: {test_loss/len(test_loader):.4f}, test Accuracy: {test_accuracy:.2f}%")

In [ ]:
plt.plot(train_losses)
plt.title("Train loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
plt.plot(train_accuracies)
plt.title("Train accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.show()

In [ ]:
plt.plot(val_losses)
plt.title("val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
plt.plot(val_accuracies)
plt.title("val accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.show()